In [1]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np
import sklearn as sk
import sklearn as sk

In [2]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

analyze_netcdf("IMD_rainfall_0p25.nc")

Analysis for NetCDF File: IMD_rainfall_0p25.nc

--- Dimensions ---
time: 31046
lat: 129
lon: 135

--- Coordinates ---
- lat:
    dtype: float64
    shape: (129,)
    attributes: {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (135,)
    attributes: {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- time:
    dtype: datetime64[ns]
    shape: (31046,)
    attributes: {'long_name': 'time', 'standard_name': 'time'}

--- Data Variables ---
- rain:
    dtype: float64
    shape: (31046, 129, 135)
    dimensions: ('time', 'lat', 'lon')
    attributes: {'long_name': 'Rainfall', 'units': 'mm/day'}

--- Global Attributes ---
Conventions: CF-1.7
comment: 
crs: epsg:4326
history: 2026-06-19 06:45:25.930728 Python
references: 
source: https://imdpune.gov.in/
title: IMD gridded data



In [3]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("s2s_reforecast_sorted.zarr")

analyze_netcdf("s2s_reforecast.nc")

Analysis for NetCDF File: s2s_reforecast.nc

--- Dimensions ---
time: 3720
step: 43
lat: 33
lon: 35

--- Coordinates ---
- lat:
    dtype: float64
    shape: (33,)
    attributes: {'long_name': 'latitude', 'standard_name': 'latitude', 'stored_direction': 'decreasing', 'units': 'degrees_north'}
- lon:
    dtype: float64
    shape: (35,)
    attributes: {'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
- step:
    dtype: int64
    shape: (43,)
- time:
    dtype: datetime64[ns]
    shape: (3720,)
    attributes: {'long_name': 'initial time of forecast', 'standard_name': 'forecast_reference_time'}

--- Data Variables ---
- 10m_u_component_of_wind:
    dtype: float32
    shape: (3720, 43, 33, 35)
    dimensions: ('time', 'step', 'lat', 'lon')
    attributes: {'GRIB_NV': np.int64(0), 'GRIB_Nx': np.int64(35), 'GRIB_Ny': np.int64(33), 'GRIB_cfName': 'eastward_wind', 'GRIB_cfVarName': 'u10', 'GRIB_dataType': 'cf', 'GRIB_gridDefinitionDescription': 'Latitude/longi

In [4]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)

In [5]:
#all libraries required
import time
from contextlib import contextmanager

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

In [ ]:
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
from openpyxl import descriptors
"""
Multi-window, lead-conditioned UNet downscaling: ECMWF S2S -> IMD 0.25 rain
anomaly over India, trained jointly across several DISJOINT forecast windows,
evaluated by rotating-year cross-validation.

WHY MULTI-WINDOW (the point of this rewrite)
--------------------------------------------
The single-window version reduced each init to one sample (3720 total) and
overfit hard -- and changing model size did nothing, which means the binding
constraint is SAMPLES, not capacity. Two facts drive the fix:

  * Adding all daily leads does NOT add real data. Day-20 and day-21 from one
    init share nearly identical predictor fields and nearly identical targets.
    43 daily leads from an init is ~3 effectively-independent samples, not 43.
    Worse, adjacent leads split across train/val is near-duplication = leak.

  * Adding DISJOINT windows DOES add data. Week-2, weeks 3-4 and weeks 5-6
    dynamics genuinely differ, and their valid-date ranges don't overlap, so
    no leak. One model correcting all three, told which window it is via an
    embedding channel, shares structure across leads -- the data-rich short
    leads regularise the data-poor long leads.

So: WINDOWS below are non-overlapping. Each (init, window) is one sample with
a window-id. Effective sample count ~3x. This is the lever that actually
moves val skill; base/dropout/weight-decay are second-order.

WHY ROTATING-YEAR CV
--------------------
20 years is few. A single 3-year val holdout is noisy -- one El Nino year in
val can dominate the estimate. Rotating leave-3-out CV over the non-test years
gives a stable mean +/- spread and uses every year for validation once.
Test years are held out of ALL folds, always.

Everything else carries over from the settled pipeline: anomaly space with
train-only climatology, strict mask, mask+lat/lon static channels,
coarse-input with in-model grid_sample upsampling, GroupNorm, masked loss.

USAGE
-----
  python s2s_unet_mw.py prepare        # archive -> cached arrays (slow, once)
  python s2s_unet_mw.py train          # rotating-year CV, then final model
  python s2s_unet_mw.py train --epochs 60 --batch 16 --base 24 --folds 5
"""
from s2s_composite_loss import CompositeLoss
import argparse
import os
import time
from contextlib import contextmanager

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"

# DISJOINT windows: (name, first_lead_day, last_lead_day) inclusive.
# Non-overlap in valid dates is what keeps the extra samples honest.
WINDOWS = [
    ("week2",   8, 14),
    ("week3_4", 15, 28),
    ("week5_6", 29, 42),
]

CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
TEST_YEARS_N = 3                # held out of every CV fold
CACHE = "unet_cache_mw.npz"
OUT_MAPS = "unet_skill_maps_mw.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE  (numpy/xarray only; torch not needed here)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...), NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Build each window's (X, y, doy) then stack, tagging window id.
    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")

            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)

            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values

            X_list.append(Xa)
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list)          # (N, V, h, w)  N = n_init * n_window
    y = np.concatenate(y_list)          # (N, H, W)
    doy = np.concatenate(doy_list)      # (N,)
    wid = np.concatenate(wid_list)      # (N,)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        # strict mask: valid on every day, per window then intersect
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose {int(np.isfinite(y).any(axis=0).sum())})")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years (held out of all folds): {sorted(test_years)}")

    with stage("Caching"):
        # NOTE: climatology + standardisation are deferred to train time,
        # because they must be recomputed per CV FOLD (train-year stats only).
        # Caching raw windowed anomable inputs would bake in a fixed split.
        np.savez_compressed(
            cache_path,
            X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clat=clat, clon=clon, flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")
        print("    (raw windowed values cached; climatology/standardisation")
        print("     done per-fold at train time so each fold is leak-free)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING  (must run per fold: train-year stats only)
# ======================================================================

def anomalise_fold(X, y, doy, tr):
    """De-climatologise both sides using TRAIN-fold years only, then
    standardise predictors on train stats. Returns processed copies.

    This is the leakage-critical step. Doing it once globally would let the
    val/test years inform the climatology; doing it per fold keeps each
    fold's estimate honest.
    """
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, H, W)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, V, h, w)

    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]

    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


# ======================================================================
# MODEL + TRAIN  (torch, lazy import)
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"])
    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    # ---- coarse->fine grid_sample grid (coordinate-aware bilinear) ----
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box; raise COARSE_PAD"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            layers = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                      nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                layers.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*layers)

        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        """Coarse predictors + window-id -> fine anomaly field.

        The window id is embedded and broadcast as an extra coarse-input
        channel, so one model corrects all windows and shares structure
        across leads. Everything else is the settled downscaling UNet:
        coarse encoder -> grid_sample to fine -> static channels -> small
        UNet refinement.
        """
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], dim=1)
            c = self.enc_c2(self.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def masked_mse(pred, target, m):
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        """Per-cell skill (vs zero-anomaly clim) and ACC over masked cells."""
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6                       # guard degenerate cells (-inf fix)
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs, tag):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        crit = CompositeLoss(w_mse=args.w_mse, w_corr=args.w_corr, w_std=args.w_std).to(dev)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss, _ = crit(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(crit(model(xb, wb, samp, stat), yb, mb)[0]) * len(xb)                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa)
        widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                p = model(Xt[j].to(dev), widt[j].to(dev), samp, stat)
                out.append(p.cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV over non-test years ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    folds = args.folds
    # contiguous blocks of years as val, rest as train
    blocks = np.array_split(nontest_years, folds)

    # resume: load any folds already completed in a prior (interrupted) run
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        import json
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming: folds {sorted(done)} already done, "
              f"skipping them ({resume_path})")

    def _save_folds(d):
        import json
        with open(resume_path, "w") as fh:
            json.dump({str(k): v for k, v in d.items()}, fh, indent=2)

    with stage(f"Rotating-year CV: {folds} folds over {len(nontest_years)} years"):
        for fi, val_years in enumerate(blocks):
            if fi in done:
                r = done[fi]
                print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | "
                      f"ACC {r['acc']:.3f}", flush=True)
                continue

            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test

            # per-fold anomalisation (train years of THIS fold only)
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs, f"fold{fi}")
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]      # mask from RAW y
            skill, acc = skill_acc(p, t, fin)
            ms = float(np.nanmean(skill[mask]))
            ma = float(np.nanmean(acc[mask]))

            # persist THIS fold immediately, before starting the next one
            done[fi] = {"skill": ms, "acc": ma, "val_years": sorted(val_years),
                        "epochs": int(eps), "vloss": float(vloss)}
            _save_folds(done)
            print(f"    fold {fi} val {sorted(val_years)}: "
                  f"skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep | vloss {vloss:.3f}"
                  f"   [saved]", flush=True)

        fold_skl = [done[i]["skill"] for i in range(folds) if i in done]
        fold_acc = [done[i]["acc"] for i in range(folds) if i in done]
        print(f"\n    CV skill {np.mean(fold_skl):+.3f} +/- {np.std(fold_skl):.3f}"
              f"   CV ACC {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

    # ---------- final model: train on ALL non-test, evaluate on test ----------
    with stage("Final model on all non-test years -> test"):
        tr_i = ~is_test
        # small val slice just for early stopping (last 2 non-test years)
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = tr_i & ~va_i

        Xa, ya, clim_y = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0],
                                  Xa, ya, args.epochs, "final")

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        # overall + per-window test skill
        import xarray as xr
        wid_te = wid[is_test]
        data_vars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sub_m = wid_te == w
            if sub_m.sum() == 0:
                continue
            sk, ac = skill_acc(p[sub_m], t[sub_m], fin[sub_m])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.3f} | "
                  f"ACC {np.nanmean(ac[mask]):.3f} | "
                  f"{100*np.nanmean(sk[mask] > 0):.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.attrs["note"] = ("anomaly-space skill = 1 - rmse/rmse_clim (clim = "
                             "zero anomaly); NaN outside IMD mask / degenerate cells")
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt). Per-window skill maps; open in Panoply.")


# ======================================================================

# Jupyter Notebook equivalent of command-line args
class Args:
    cmd = 'train' # Set to 'prepare' to generate cache, or 'train' to train the model
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5
    w_mse = 1.0
    w_corr = 1.0
    w_std = 10.0

args = Args()

if args.cmd == 'prepare':
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
    imd_ds = ds_imd      # noqa: F821
    prepare(ecmwf_ds, imd_ds, CACHE)
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f'run `prepare` first ({CACHE} missing)')
    build_and_run(args, CACHE, OUT_MAPS)


    device: mps
    resuming: folds [0, 1, 2, 3, 4] already done, skipping them (unet_skill_maps_mw_folds.json)
[ ] Rotating-year CV: 5 folds over 17 years ...
    fold 0 (cached): skill +0.025 | ACC 0.246
    fold 1 (cached): skill +0.030 | ACC 0.242
    fold 2 (cached): skill +0.029 | ACC 0.260
    fold 3 (cached): skill +0.029 | ACC 0.260
    fold 4 (cached): skill +0.026 | ACC 0.239

    CV skill +0.028 +/- 0.002   CV ACC 0.249 +/- 0.009
[x] Rotating-year CV: 5 folds over 17 years  (0.0s)
[ ] Final model on all non-test years -> test ...


KeyboardInterrupt: 


Amplitude-shrinkage check (test samples)
  window |  obs~pred slope |  pred~obs slope |  std ratio |    ACC
         | (1=ok, >1 shrink) | (1=ok, <1 shrink) |        p/o |
   week2 |           1.113 |           0.118 |      0.325 |  0.362
 week3_4 |           0.840 |           0.044 |      0.229 |  0.193
 week5_6 |           0.760 |           0.017 |      0.149 |  0.114

-> shrinkage_scatter.png

Verdict per window:
     week2: obs~pred slope 1.11, std ratio 0.33 -- SHRINKAGE (predicts ~33% of true amplitude) -> loss is the lever, not architecture
   week3_4: obs~pred slope 0.84, std ratio 0.23 -- SHRINKAGE (predicts ~23% of true amplitude) -> loss is the lever, not architecture
   week5_6: obs~pred slope 0.76, std ratio 0.15 -- SHRINKAGE (predicts ~15% of true amplitude) -> loss is the lever, not architecture


# train script

In [14]:
"""
Multi-window, lead-conditioned UNet downscaling: ECMWF S2S -> IMD 0.25 rain
anomaly over India, trained jointly across several DISJOINT forecast windows,
evaluated by rotating-year cross-validation.

WHY MULTI-WINDOW (the point of this rewrite)
--------------------------------------------
The single-window version reduced each init to one sample (3720 total) and
overfit hard -- and changing model size did nothing, which means the binding
constraint is SAMPLES, not capacity. Two facts drive the fix:

  * Adding all daily leads does NOT add real data. Day-20 and day-21 from one
    init share nearly identical predictor fields and nearly identical targets.
    43 daily leads from an init is ~3 effectively-independent samples, not 43.
    Worse, adjacent leads split across train/val is near-duplication = leak.

  * Adding DISJOINT windows DOES add data. Week-2, weeks 3-4 and weeks 5-6
    dynamics genuinely differ, and their valid-date ranges don't overlap, so
    no leak. One model correcting all three, told which window it is via an
    embedding channel, shares structure across leads -- the data-rich short
    leads regularise the data-poor long leads.

So: WINDOWS below are non-overlapping. Each (init, window) is one sample with
a window-id. Effective sample count ~3x. This is the lever that actually
moves val skill; base/dropout/weight-decay are second-order.

WHY ROTATING-YEAR CV
--------------------
20 years is few. A single 3-year val holdout is noisy -- one El Nino year in
val can dominate the estimate. Rotating leave-3-out CV over the non-test years
gives a stable mean +/- spread and uses every year for validation once.
Test years are held out of ALL folds, always.

Everything else carries over from the settled pipeline: anomaly space with
train-only climatology, strict mask, mask+lat/lon static channels,
coarse-input with in-model grid_sample upsampling, GroupNorm, masked loss.

USAGE
-----
  python s2s_unet_mw.py prepare        # archive -> cached arrays (slow, once)
  python s2s_unet_mw.py train          # rotating-year CV, then final model
  python s2s_unet_mw.py train --epochs 60 --batch 16 --base 24 --folds 5
"""
from s2s_composite_loss import CompositeLoss

import argparse
import os
import time
from contextlib import contextmanager

import numpy as np

# ---- CONFIG ----
IMD_TARGET_VAR = "rain"

# DISJOINT windows: (name, first_lead_day, last_lead_day) inclusive.
# Non-overlap in valid dates is what keeps the extra samples honest.
WINDOWS = [
    ("week2",   8, 14),
    ("week3_4", 15, 28),
    ("week5_6", 29, 42),
]

# Jupyter Notebook equivalent of command-line args
class Args:
    cmd = 'train' # Set to 'prepare' to generate cache, or 'train' to train the model
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5
    w_mse = 1.0
    w_corr = 1.0
    w_std = 10.0

args = Args()





CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                   # e.g. (6, 7, 8, 9) for JJAS inits
TEST_YEARS_N = 3                # held out of every CV fold
CACHE = "unet_cache_mw.npz"
OUT_MAPS = f"unet_mw_mse{args.w_mse}_corr{args.w_corr}_std{args.w_std}.nc"
DTYPE = np.float32


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE  (numpy/xarray only; torch not needed here)
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    """(n, ...) -> (366, ...), NaN-aware DOY climatology via matmul."""
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    # Build each window's (X, y, doy) then stack, tagging window id.
    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")

            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)

            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)

            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values

            X_list.append(Xa)
            y_list.append(ya)
            doy_list.append(doya)
            wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list)          # (N, V, h, w)  N = n_init * n_window
    y = np.concatenate(y_list)          # (N, H, W)
    doy = np.concatenate(doy_list)      # (N,)
    wid = np.concatenate(wid_list)      # (N,)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        # strict mask: valid on every day, per window then intersect
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells "
              f"(loose {int(np.isfinite(y).any(axis=0).sum())})")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years (held out of all folds): {sorted(test_years)}")

    with stage("Caching"):
        # NOTE: climatology + standardisation are deferred to train time,
        # because they must be recomputed per CV FOLD (train-year stats only).
        # Caching raw windowed anomable inputs would bake in a fixed split.
        np.savez_compressed(
            cache_path,
            X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test,
            clat=clat, clon=clon, flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]),
        )
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")
        print("    (raw windowed values cached; climatology/standardisation")
        print("     done per-fold at train time so each fold is leak-free)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING  (must run per fold: train-year stats only)
# ======================================================================

def anomalise_fold(X, y, doy, tr):
    """De-climatologise both sides using TRAIN-fold years only, then
    standardise predictors on train stats. Returns processed copies.

    This is the leakage-critical step. Doing it once globally would let the
    val/test years inform the climatology; doing it per fold keeps each
    fold's estimate honest.
    """
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, H, W)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)      # (366, V, h, w)

    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]

    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


# ======================================================================
# MODEL + TRAIN  (torch, lazy import)
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")
    print(f"    device: {dev}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"])
    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    # ---- coarse->fine grid_sample grid (coordinate-aware bilinear) ----
    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box; raise COARSE_PAD"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            layers = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                      nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                layers.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*layers)

        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        """Coarse predictors + window-id -> fine anomaly field.

        The window id is embedded and broadcast as an extra coarse-input
        channel, so one model corrects all windows and shares structure
        across leads. Everything else is the settled downscaling UNet:
        coarse encoder -> grid_sample to fine -> static channels -> small
        UNet refinement.
        """
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], dim=1)
            c = self.enc_c2(self.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], dim=1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f)
            e1 = self.d1(self.pool(e0))
            e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], dim=1))
            u = self.du1(torch.cat([self.u1(u), e0], dim=1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def masked_mse(pred, target, m):
        se = (pred - target) ** 2 * m
        return se.sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        """Per-cell skill (vs zero-anomaly clim) and ACC over masked cells."""
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6                       # guard degenerate cells (-inf fix)
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs, tag):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                                weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        crit = CompositeLoss(w_mse=args.w_mse, w_corr=args.w_corr, w_std=args.w_std).to(dev)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss, _ = crit(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(crit(model(xb, wb, samp, stat), yb, mb)[0]) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa)
        widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                p = model(Xt[j].to(dev), widt[j].to(dev), samp, stat)
                out.append(p.cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV over non-test years ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    folds = args.folds
    # contiguous blocks of years as val, rest as train
    blocks = np.array_split(nontest_years, folds)

    # resume: load any folds already completed in a prior (interrupted) run
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        import json
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming: folds {sorted(done)} already done, "
              f"skipping them ({resume_path})")

    def _save_folds(d):
        import json
        with open(resume_path, "w") as fh:
            json.dump({str(k): v for k, v in d.items()}, fh, indent=2)

    with stage(f"Rotating-year CV: {folds} folds over {len(nontest_years)} years"):
        for fi, val_years in enumerate(blocks):
            if fi in done:
                r = done[fi]
                print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | "
                      f"ACC {r['acc']:.3f}", flush=True)
                continue

            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test

            # per-fold anomalisation (train years of THIS fold only)
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs, f"fold{fi}")
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]      # mask from RAW y
            skill, acc = skill_acc(p, t, fin)
            ms = float(np.nanmean(skill[mask]))
            ma = float(np.nanmean(acc[mask]))

            # persist THIS fold immediately, before starting the next one
            done[fi] = {"skill": ms, "acc": ma, "val_years": sorted(val_years),
                        "epochs": int(eps), "vloss": float(vloss)}
            _save_folds(done)
            print(f"    fold {fi} val {sorted(val_years)}: "
                  f"skill {ms:+.3f} | ACC {ma:.3f} | {eps} ep | vloss {vloss:.3f}"
                  f"   [saved]", flush=True)

        fold_skl = [done[i]["skill"] for i in range(folds) if i in done]
        fold_acc = [done[i]["acc"] for i in range(folds) if i in done]
        print(f"\n    CV skill {np.mean(fold_skl):+.3f} +/- {np.std(fold_skl):.3f}"
              f"   CV ACC {np.mean(fold_acc):.3f} +/- {np.std(fold_acc):.3f}")

    # ---------- final model: train on ALL non-test, evaluate on test ----------
    with stage("Final model on all non-test years -> test"):
        tr_i = ~is_test
        # small val slice just for early stopping (last 2 non-test years)
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = tr_i & ~va_i

        Xa, ya, clim_y = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0],
                                  Xa, ya, args.epochs, "final")

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        # overall + per-window test skill
        import xarray as xr
        wid_te = wid[is_test]
        data_vars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sub_m = wid_te == w
            if sub_m.sum() == 0:
                continue
            sk, ac = skill_acc(p[sub_m], t[sub_m], fin[sub_m])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.3f} | "
                  f"ACC {np.nanmean(ac[mask]):.3f} | "
                  f"{100*np.nanmean(sk[mask] > 0):.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.attrs["note"] = ("anomaly-space skill = 1 - rmse/rmse_clim (clim = "
                             "zero anomaly); NaN outside IMD mask / degenerate cells")
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": vars(args)},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ .pt). Per-window skill maps; open in Panoply.")


# ======================================================================

if args.cmd == 'prepare':
    ecmwf_ds = ds_ecmv   # noqa: F821  <- replace with your open datasets
    imd_ds = ds_imd      # noqa: F821
    prepare(ecmwf_ds, imd_ds, CACHE)
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f'run `prepare` first ({CACHE} missing)')
    build_and_run(args, CACHE, OUT_MAPS)


    device: mps
[ ] Rotating-year CV: 5 folds over 17 years ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/3099982593.py:365: RuntimeWarning: Mean of empty slice
  rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/3099982593.py:366: RuntimeWarning: Mean of empty slice
  rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/3099982593.py:369: RuntimeWarning: Mean of empty slice
  tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/3099982593.py:370: RuntimeWarning: Mean of empty slice
  pm = np.nanmean(np.where(fin, p, np.nan), axis=0)


    fold 0 val [2005, 2006, 2007, 2008]: skill +0.001 | ACC 0.220 | 22 ep | vloss 31.782   [saved]


KeyboardInterrupt: 

# model comparo


In [ ]:
"""
Three-line comparison per window: ECMWF forecast vs UNet vs observed IMD.

Plots the India-mean anomaly time series over the test years, one line each:
    observed IMD, raw ECMWF, UNet.

Anomalies (each minus its own seasonal climatology), because raw ECMWF tp and
IMD rain are in different units -- anomalies put all three on one axis.

Reads the same cache + checkpoint. No retraining.
  python s2s_threeline.py
"""

import argparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

DTYPE = np.float32
CLIM_WINDOW_DAYS = 7
RAW_PRECIP_VAR = "total_precipitation"


def _clim(values, doys, window=CLIM_WINDOW_DAYS, n_doy=366):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    M = (np.minimum(d, n_doy - d) <= window).astype(DTYPE)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return c.reshape((n_doy,) + shp).astype(DTYPE)


def unet_predict(args, z, X, y, doy, wid, fit_i, clat, clon, flat_lat, flat_lon, mask):
    import torch, torch.nn as nn, torch.nn.functional as F
    H, W = len(flat_lat), len(flat_lon)
    n_var, n_win = X.shape[1], len(z["window_names"])
    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")

    clim_X = _clim(X[fit_i], doy[fit_i])
    Xa = X - clim_X[doy - 1]
    xm = np.nanmean(Xa[fit_i], (0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[fit_i], (0, 2, 3), keepdims=True)
    Xa = np.nan_to_num((Xa - xm) / np.where(xs < 1e-8, 1.0, xs)).astype(DTYPE)

    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp = torch.tensor(np.stack([gxx, gyy], -1).astype(np.float32)[None]).to(dev)
    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = torch.tensor(np.stack([mask.astype(DTYPE),
        np.broadcast_to(lat2, (H, W)).astype(DTYPE),
        np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]).to(dev)

    ckpt = torch.load(args.ckpt, map_location=dev)
    base, drop = ckpt["args"].get("base", 24), ckpt["args"].get("drop", 0.2)

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(s, ci, co, dr=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if dr > 0: L.append(nn.Dropout2d(dr))
            s.f = nn.Sequential(*L)
        def forward(s, x): return s.f(x)

    class MWUNet(nn.Module):
        def __init__(s, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            s.emb = nn.Embedding(n_win, emb)
            s.enc_c1 = Block(n_var + emb, base*2); s.enc_c2 = Block(base*2, base*2)
            s.inp = Block(base*2+3, base); s.d1 = Block(base, base*2, drop); s.d2 = Block(base*2, base*4, drop)
            s.bott = Block(base*4, base*4, drop)
            s.u2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); s.du2 = Block(base*4, base*2, drop)
            s.u1 = nn.ConvTranspose2d(base*2, base, 2, stride=2); s.du1 = Block(base*2, base)
            s.head = nn.Conv2d(base, 1, 1); s.pool = nn.MaxPool2d(2)
        def forward(s, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = s.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            c = s.enc_c2(s.enc_c1(torch.cat([xc, e], 1)))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = s.inp(f); e1 = s.d1(s.pool(e0)); e2 = s.d2(s.pool(e1))
            u = s.du2(torch.cat([s.u2(s.bott(e2)), e1], 1))
            u = s.du1(torch.cat([s.u1(u), e0], 1))
            return s.head(u)[:, :, :H0, :W0].squeeze(1)

    model = MWUNet(n_var, n_win, base=base, drop=drop).to(dev)
    model.load_state_dict(ckpt["state"]); model.eval()
    Xt, widt = torch.tensor(Xa), torch.tensor(wid)
    out = []
    with torch.no_grad():
        for a in range(0, len(X), args.batch):
            j = slice(a, a + args.batch)
            out.append(model(Xt[j].to(dev), widt[j].to(dev), samp, static).cpu().numpy())
    return np.concatenate(out)


def main():
    class Args:
        cache = "unet_cache_mw.npz"
        ckpt = "unet_skill_maps_mw.pt"
        batch = 16
        out = "threeline.png"
    args = Args()


    z = np.load(args.cache, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    mask, is_test, year = z["mask"], z["is_test"], z["year"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    feature_vars = [str(v) for v in z["feature_vars"]]
    window_names = [str(w) for w in z["window_names"]]
    n_win = len(window_names)
    tp_idx = feature_vars.index(RAW_PRECIP_VAR)
    fit_i = ~is_test

    # raw ECMWF tp on the fine grid
    import xarray as xr
    tp_fine = xr.DataArray(X[:, tp_idx], dims=("s", "lat", "lon"),
                           coords={"lat": clat, "lon": clon}
                           ).interp(lat=flat_lat, lon=flat_lon).values.astype(DTYPE)

    # anomalies, each vs its own train climatology
    obs_a = y - _clim(y[fit_i], doy[fit_i])[doy - 1]
    ecm_a = tp_fine - _clim(tp_fine[fit_i], doy[fit_i])[doy - 1]
    unet_a = unet_predict(args, z, X, y, doy, wid, fit_i,
                          clat, clon, flat_lat, flat_lon, mask)

    # India-mean over valid cells, test samples only
    m3 = mask[None]
    def india_mean(a):
        return np.nansum(np.where(m3, a, np.nan) * m3, axis=(1, 2)) / m3.sum()

    fig, axes = plt.subplots(n_win, 1, figsize=(13, 3.2 * n_win), sharex=False)
    axes = np.atleast_1d(axes)
    for w in range(n_win):
        sel = (wid == w) & is_test
        order = np.argsort(np.arange(len(sel))[sel])   # keep chronological-ish
        idx = np.where(sel)[0]
        ax = axes[w]
        t = np.arange(len(idx))
        ax.plot(t, india_mean(obs_a)[idx], "-", color="black", lw=2, label="observed IMD")
        ax.plot(t, india_mean(ecm_a)[idx], "-", color="#888", lw=1.5, label="raw ECMWF")
        ax.plot(t, india_mean(unet_a)[idx], "-", color="#E45756", lw=1.5, label="UNet")
        ax.axhline(0, color="gray", lw=0.5, ls=":")
        r_u = np.corrcoef(india_mean(unet_a)[idx], india_mean(obs_a)[idx])[0, 1]
        r_e = np.corrcoef(india_mean(ecm_a)[idx], india_mean(obs_a)[idx])[0, 1]
        ax.set_title(f"{window_names[w]}  (India-mean anomaly; "
                     f"corr vs obs: UNet {r_u:.2f}, ECMWF {r_e:.2f})")
        ax.set_ylabel("anomaly (mm/day)")
        if w == 0:
            ax.legend(ncol=3, loc="upper right", fontsize=9)
    axes[-1].set_xlabel("test-set forecast index (chronological within window)")
    fig.tight_layout()
    fig.savefig(args.out, dpi=140)
    print(f"-> {args.out}")


if __name__ == "__main__":
    main()

# shrinkage check

In [ ]:
"""
Amplitude-shrinkage diagnostic: predicted vs observed anomaly, per window.

Tests one specific hypothesis: is MSE training making the UNet hedge
amplitudes (regress toward the mean)? If so, the model has real pattern
skill (nonzero ACC) but damped magnitudes, which craters RMSE skill. That
would mean the fix is the LOSS, not the architecture.

HOW TO READ IT (the key is the RELIABILITY SLOPE)
-------------------------------------------------
Regress OBSERVED on PREDICTED (obs = a + b*pred):
    b ~ 1   -> calibrated amplitudes, no shrinkage. MSE is not the problem.
    b > 1   -> SHRINKAGE. Observed swings more than the model dares to
               predict; the net hedges. b = 2 means it predicts half the
               true amplitude. This is the MSE-toward-the-mean signature.
    b < 1   -> over-confident (rare for MSE training).

Also shown, so it's unambiguous:
  * the reverse slope (pred on obs) -- for MSE shrinkage this is < 1
  * the 1:1 diagonal
  * a binned conditional-mean curve E[obs | pred] -- catches nonlinear
    shrinkage a single OLS line would hide (e.g. shrinks only in the tails)
  * amplitude ratio std(pred)/std(obs) -- a scale-free shrinkage number

Reads the same cache + checkpoint as the eval script. No retraining.

USAGE
  python s2s_shrinkage.py
  python s2s_shrinkage.py --cache unet_cache_mw.npz --ckpt unet_skill_maps_mw.pt
"""

import argparse

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

DTYPE = np.float32
CLIM_WINDOW_DAYS = 7


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim(values, doys, window):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        c = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return c.reshape((366,) + shp).astype(DTYPE)


def ols(x, y):
    """slope, intercept, r for y ~ a + b x, over finite pairs."""
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 10:
        return np.nan, np.nan, np.nan, 0
    b, a = np.polyfit(x, y, 1)
    r = np.corrcoef(x, y)[0, 1]
    return b, a, r, len(x)


def run_unet_predictions(args, z, X, y, doy, wid, fit_i,
                         clat, clon, flat_lat, flat_lon, mask):
    """Reload checkpoint, reproduce fit-year anomaly space, predict ALL samples.
    (Identical preprocessing to training's final model.)"""
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]
    n_win = len(z["window_names"])
    dev = ("mps" if torch.backends.mps.is_available()
           else "cuda" if torch.cuda.is_available() else "cpu")

    clim_X = _clim(X[fit_i], doy[fit_i], CLIM_WINDOW_DAYS)
    Xa = X - clim_X[doy - 1]
    xm = np.nanmean(Xa[fit_i], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[fit_i], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs).astype(DTYPE)

    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp = torch.tensor(np.stack([gxx, gyy], -1).astype(np.float32)[None]).to(dev)
    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = torch.tensor(np.stack([
        mask.astype(DTYPE),
        np.broadcast_to(lat2, (H, W)).astype(DTYPE),
        np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]).to(dev)

    ckpt = torch.load(args.ckpt, map_location=dev)
    base = ckpt["args"].get("base", 24)
    drop = ckpt["args"].get("drop", 0.2)

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(s, ci, co, dr=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if dr > 0:
                L.append(nn.Dropout2d(dr))
            s.f = nn.Sequential(*L)

        def forward(s, x):
            return s.f(x)

    class MWUNet(nn.Module):
        def __init__(s, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            s.emb = nn.Embedding(n_win, emb)
            s.enc_c1 = Block(n_var + emb, base * 2); s.enc_c2 = Block(base * 2, base * 2)
            s.inp = Block(base * 2 + 3, base)
            s.d1 = Block(base, base * 2, drop); s.d2 = Block(base * 2, base * 4, drop)
            s.bott = Block(base * 4, base * 4, drop)
            s.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2); s.du2 = Block(base * 4, base * 2, drop)
            s.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2); s.du1 = Block(base * 2, base)
            s.head = nn.Conv2d(base, 1, 1); s.pool = nn.MaxPool2d(2)

        def forward(s, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = s.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            xc = torch.cat([xc, e], 1)
            c = s.enc_c2(s.enc_c1(xc))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = s.inp(f); e1 = s.d1(s.pool(e0)); e2 = s.d2(s.pool(e1))
            u = s.du2(torch.cat([s.u2(s.bott(e2)), e1], 1))
            u = s.du1(torch.cat([s.u1(u), e0], 1))
            return s.head(u)[:, :, :H0, :W0].squeeze(1)

    model = MWUNet(n_var, n_win, base=base, drop=drop).to(dev)
    model.load_state_dict(ckpt["state"])
    model.eval()

    Xt = torch.tensor(Xa)
    widt = torch.tensor(wid)
    out = []
    with torch.no_grad():
        for a in range(0, len(X), args.batch):
            j = slice(a, a + args.batch)
            out.append(model(Xt[j].to(dev), widt[j].to(dev), samp, static).cpu().numpy())
    return np.concatenate(out)


def main():
    class Args:
        cache = 'unet_cache_mw.npz'
        ckpt = 'unet_skill_maps_mw.pt'
        batch = 16
        out = 'shrinkage'
        split = 'test'
    args = Args()

    z = np.load(args.cache, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    mask, is_test = z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    window_names = [str(w) for w in z["window_names"]]
    n_win = len(window_names)

    fit_i = ~is_test
    pred_all = run_unet_predictions(args, z, X, y, doy, wid, fit_i,
                                    clat, clon, flat_lat, flat_lon, mask)

    clim_y = _clim(y[fit_i], doy[fit_i], CLIM_WINDOW_DAYS)
    obs_anom_all = y - clim_y[doy - 1]

    eval_mask = is_test if args.split == "test" else np.ones(len(y), bool)

    fig, axes = plt.subplots(1, n_win, figsize=(6 * n_win, 5.6))
    axes = np.atleast_1d(axes)
    print(f"\nAmplitude-shrinkage check ({args.split} samples)")
    print(f"{'window':>8} | {'obs~pred slope':>15} | {'pred~obs slope':>15} "
          f"| {'std ratio':>10} | {'ACC':>6}")
    print(f"{'':>8} | {'(1=ok, >1 shrink)':>15} | {'(1=ok, <1 shrink)':>15} "
          f"| {'p/o':>10} |")

    for w in range(n_win):
        sel = (wid == w) & eval_mask
        fin = np.isfinite(obs_anom_all) & mask[None]
        cell = fin[sel]                       # (n_w, H, W)
        pr = pred_all[sel][cell]              # flat valid predicted anomalies
        ob = obs_anom_all[sel][cell]          # flat valid observed anomalies

        b_op, a_op, r, n = ols(pr, ob)        # observed ~ predicted (reliability)
        b_po, _, _, _ = ols(ob, pr)           # predicted ~ observed (reverse)
        std_ratio = np.std(pr) / np.std(ob) if np.std(ob) > 0 else np.nan

        print(f"{window_names[w]:>8} | {b_op:>15.3f} | {b_po:>15.3f} "
              f"| {std_ratio:>10.3f} | {r:>6.3f}")

        ax = axes[w]
        # subsample for the scatter (millions of points otherwise)
        if len(pr) > 40000:
            idx = np.random.default_rng(0).choice(len(pr), 40000, replace=False)
            ps, os_ = pr[idx], ob[idx]
        else:
            ps, os_ = pr, ob
        ax.scatter(ps, os_, s=2, alpha=0.06, color="#4C78A8", rasterized=True)

        lim = np.nanpercentile(np.abs(np.concatenate([pr, ob])), 99)
        xs_line = np.linspace(-lim, lim, 100)
        ax.plot(xs_line, xs_line, "k-", lw=1.2, label="1:1 (calibrated)")
        ax.plot(xs_line, b_op * xs_line + a_op, "r--", lw=1.8,
                label=f"obs~pred  (slope {b_op:.2f})")

        # binned conditional mean E[obs | pred] -- catches nonlinear shrinkage
        bins = np.linspace(-lim, lim, 16)
        who = np.digitize(pr, bins)
        cx, cy = [], []
        for bi in range(1, len(bins)):
            m = who == bi
            if m.sum() > 30:
                cx.append(pr[m].mean()); cy.append(ob[m].mean())
        ax.plot(cx, cy, "o-", color="#E45756", lw=1.5, ms=4,
                label="E[obs | pred]")

        ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_xlabel("predicted anomaly (mm/day)")
        ax.set_ylabel("observed anomaly (mm/day)")
        ax.set_title(f"{window_names[w]}  (ACC {r:.2f}, std ratio {std_ratio:.2f})")
        ax.legend(fontsize=8, loc="upper left")
        ax.set_aspect("equal", "box")

    fig.tight_layout()
    fig.savefig(f"{args.out}_scatter.png", dpi=130)
    plt.close(fig)
    print(f"\n-> {args.out}_scatter.png")

    print("\nVerdict per window:")
    for w in range(n_win):
        sel = (wid == w) & eval_mask
        fin = np.isfinite(obs_anom_all) & mask[None]
        cell = fin[sel]
        pr, ob = pred_all[sel][cell], obs_anom_all[sel][cell]
        b_op, *_ = ols(pr, ob)
        sr = np.std(pr) / np.std(ob) if np.std(ob) > 0 else np.nan
        if b_op > 1.3 or sr < 0.77:
            v = (f"SHRINKAGE (predicts ~{sr:.0%} of true amplitude) -> "
                 "loss is the lever, not architecture")
        elif b_op < 0.85:
            v = "over-confident amplitudes (unusual for MSE)"
        else:
            v = "amplitudes ~calibrated -> shrinkage is NOT the story here"
        print(f"  {window_names[w]:>8}: obs~pred slope {b_op:.2f}, "
              f"std ratio {sr:.2f} -- {v}")

if __name__ == "__main__":
    main()

In [ ]:
# ======================================================================
# S2S multi-window UNet -- LOSS-FAMILY SWEEP VERSION
# Single-cell Jupyter script. Set LOSS_NAME below, run, compare.
#
# Losses available (all masked, all anomaly-space):
#   "mse"          plain masked MSE                     -> conditional MEAN
#   "mae"          masked L1                            -> conditional MEDIAN
#   "huber"        smooth-L1, delta-tunable             -> robust mean/median blend
#   "wmse"         intensity-weighted MSE (Chandel-style extremes weighting)
#   "tail"         tail-weighted: MSE * (1 + a*|y|/std)^p
#   "pinball"      asymmetric quantile loss at one tau (median-ish, tilted)
#   "logcosh"      smooth, MAE-like tails, MSE-like centre
#   "grad"         MSE + spatial-gradient matching (structure, not amplitude)
#   "spectral"     MSE + radial power-spectrum matching (sharpness/structure)
#
# NOTE ON EXPECTATIONS: mse/mae/huber/wmse/tail/pinball/logcosh are all
# POINTWISE -- each converges to some conditional central statistic, so they
# tend to sit on the same skill/ACC frontier and mostly trade WHICH errors
# they tolerate, not how much total signal they extract. "grad" and
# "spectral" are the two that score spatial STRUCTURE rather than per-cell
# error, so they are the ones with a mechanism to move skill and ACC
# together. Run them last if the pointwise family disappoints.
# ======================================================================

import os, time, json
from contextlib import contextmanager
import numpy as np

# ---------------- CONFIG ----------------
LOSS_NAME = "mae"          # <-- change this per run
LOSS_KW = {}               # per-loss knobs, e.g. {"delta": 1.0} for huber
                           #   huber   : delta (default 1.0)
                           #   wmse    : power (default 1.0)
                           #   tail    : alpha (default 1.0), power (default 1.0)
                           #   pinball : tau (default 0.5)
                           #   grad    : w_grad (default 1.0)
                           #   spectral: w_spec (default 1.0)

IMD_TARGET_VAR = "rain"
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None
TEST_YEARS_N = 3
CACHE = "unet_cache_mw.npz"
DTYPE = np.float32

class Args:
    cmd = "train"          # "prepare" or "train"
    epochs = 60
    batch = 16
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5

args = Args()
ARG_DICT = {k: getattr(args, k) for k in dir(args) if not k.startswith("_")}
ARG_DICT["loss"] = LOSS_NAME
ARG_DICT["loss_kw"] = LOSS_KW

# separate outputs per loss so runs never collide or resume each other
tag = LOSS_NAME + ("_" + "_".join(f"{k}{v}" for k, v in LOSS_KW.items()) if LOSS_KW else "")
OUT_MAPS = f"unet_mw_{tag}.nc"


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# PREPARE
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _doy_matrix(doys, window, n_doy=366):
    centers = np.arange(1, n_doy + 1)
    d = np.abs(doys[None, :].astype(int) - centers[:, None])
    return (np.minimum(d, n_doy - d) <= window).astype(DTYPE)


def _clim_grid(values, doys, window):
    shp = values.shape[1:]
    v2 = values.reshape(len(values), -1)
    M = _doy_matrix(doys, window)
    counts = M @ np.isfinite(v2).astype(DTYPE)
    sums = M @ np.nan_to_num(v2).astype(DTYPE)
    with np.errstate(invalid="ignore", divide="ignore"):
        clim = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    return clim.reshape((366,) + shp).astype(DTYPE)


def prepare(ecmwf_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    with stage("Coarse subset over padded India box"):
        ecmwf_ds = ensure_valid_time(ecmwf_ds)
        for c in ("lat", "lon"):
            if ecmwf_ds[c].values[0] > ecmwf_ds[c].values[-1]:
                ecmwf_ds = ecmwf_ds.sortby(c)
        la0, la1 = float(imd_ds.lat.min()), float(imd_ds.lat.max())
        lo0, lo1 = float(imd_ds.lon.min()), float(imd_ds.lon.max())
        sub = ecmwf_ds.sel(lat=slice(la0 - COARSE_PAD, la1 + COARSE_PAD),
                           lon=slice(lo0 - COARSE_PAD, lo1 + COARSE_PAD))
        if MONTHS is not None:
            sub = sub.sel(time=sub["time"].dt.month.isin(list(MONTHS)))
        feature_vars = list(sub.data_vars)
        clat, clon = sub["lat"].values, sub["lon"].values
        leads = (sub["step"].values / np.timedelta64(1, "D")).astype(int)
        init = sub["time"].values
        print(f"    {dict(sub.sizes)} x {len(feature_vars)} vars")

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    X_list, y_list, doy_list, wid_list = [], [], [], []
    for wid, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi})"):
            sel = np.where((leads >= lo) & (leads <= hi))[0]
            if len(sel) == 0:
                raise ValueError(f"no leads in [{lo},{hi}]")
            with ProgressBar():
                Xw = sub.isel(step=sel).mean(dim="step").compute()
            Xa = np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE)
            vt = sub["valid_time"].isel(step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            ya = np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1)
            centre = init + np.timedelta64((lo + hi) // 2, "D")
            doya = xr.DataArray(centre, dims="t").dt.dayofyear.values
            X_list.append(Xa); y_list.append(ya)
            doy_list.append(doya); wid_list.append(np.full(len(Xa), wid, dtype=np.int64))
            print(f"    +{len(Xa)} samples")

    X = np.concatenate(X_list); y = np.concatenate(y_list)
    doy = np.concatenate(doy_list); wid = np.concatenate(wid_list)
    year = np.concatenate([init.astype("datetime64[Y]").astype(int) + 1970] * len(WINDOWS))
    print(f"\n    total {len(X)} samples ({len(init)} inits x {len(WINDOWS)} windows)")

    with stage("Strict mask + test-year holdout"):
        mask = np.isfinite(y).all(axis=0)
        print(f"    strict mask: {int(mask.sum())} cells")
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    test years: {sorted(test_years)}")

    with stage("Caching"):
        np.savez_compressed(
            cache_path, X=X, y=y, doy=doy, wid=wid, year=year,
            mask=mask, is_test=is_test, clat=clat, clon=clon,
            flat_lat=flat_lat, flat_lon=flat_lon,
            feature_vars=np.array(feature_vars),
            window_names=np.array([w[0] for w in WINDOWS]))
        print(f"    {os.path.getsize(cache_path)/1e9:.2f} GB -> {cache_path}")


def anomalise_fold(X, y, doy, tr):
    """Train-fold-only climatology + standardisation. Leakage-critical."""
    clim_y = _clim_grid(y[tr], doy[tr], CLIM_WINDOW_DAYS)
    clim_X = _clim_grid(X[tr], doy[tr], CLIM_WINDOW_DAYS)
    ya = y - clim_y[doy - 1]
    Xa = X - clim_X[doy - 1]
    xm = np.nanmean(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.nanstd(Xa[tr], axis=(0, 2, 3), keepdims=True)
    xs = np.where(xs < 1e-8, 1.0, xs)
    Xa = np.nan_to_num((Xa - xm) / xs)
    return Xa.astype(DTYPE), ya.astype(DTYPE), clim_y


# ======================================================================
# LOSS FAMILY
# ======================================================================

def make_loss(name, target_std, **kw):
    """Return fn(pred, target, mask) -> scalar. All masked, all normalised
    by mask.sum() (never numel), all operating on anomalies.

    target_std: scalar float, train-set anomaly std. Used to make the
    intensity-weighted variants scale-free so their knobs mean the same
    thing regardless of units.
    """
    import torch
    import torch.nn.functional as F

    def _m(x, m):
        """mask, sum, normalise by mask -- the one correct reduction."""
        return (x * m).sum() / m.sum().clamp(min=1.0)

    if name == "mse":
        def f(p, t, m):
            return _m((p - t) ** 2, m)

    elif name == "mae":
        def f(p, t, m):
            return _m((p - t).abs(), m)

    elif name == "huber":
        delta = kw.get("delta", 1.0)
        def f(p, t, m):
            e = (p - t).abs()
            q = torch.clamp(e, max=delta)
            return _m(0.5 * q ** 2 + delta * (e - q), m)

    elif name == "logcosh":
        def f(p, t, m):
            e = p - t
            # numerically stable log(cosh(e))
            return _m(e + F.softplus(-2.0 * e) - float(np.log(2.0)), m) \
                if False else _m(torch.log(torch.cosh(torch.clamp(e, -12, 12))), m)

    elif name == "wmse":
        # intensity-weighted MSE: weight each cell by how extreme the OBS is.
        # This is the Chandel-style "weighted loss for extremes" idea.
        power = kw.get("power", 1.0)
        def f(p, t, m):
            w = 1.0 + (t.abs() / target_std) ** power
            return _m(w * (p - t) ** 2, m)

    elif name == "tail":
        # like wmse but with an explicit strength knob; alpha=0 -> plain MSE
        alpha = kw.get("alpha", 1.0)
        power = kw.get("power", 1.0)
        def f(p, t, m):
            w = (1.0 + alpha * (t.abs() / target_std)) ** power
            return _m(w * (p - t) ** 2, m)

    elif name == "pinball":
        # asymmetric: under-prediction costs tau/(1-tau) more than over.
        # tau=0.5 == MAE/2. tau>0.5 pushes predictions UP (anti-hedge for wet).
        tau = kw.get("tau", 0.5)
        def f(p, t, m):
            e = t - p
            return _m(torch.maximum(tau * e, (tau - 1.0) * e), m)

    elif name == "grad":
        # MSE + spatial-gradient matching. Scores STRUCTURE, not per-cell
        # amplitude, so it has a mechanism the pointwise family lacks.
        wg = kw.get("w_grad", 1.0)
        def f(p, t, m):
            base = _m((p - t) ** 2, m)
            mx = m[:, :, 1:] * m[:, :, :-1]
            my = m[:, 1:, :] * m[:, :-1, :]
            gx = ((p[:, :, 1:] - p[:, :, :-1]) - (t[:, :, 1:] - t[:, :, :-1])) ** 2
            gy = ((p[:, 1:, :] - p[:, :-1, :]) - (t[:, 1:, :] - t[:, :-1, :])) ** 2
            return base + wg * (_m(gx, mx) + _m(gy, my))

    elif name == "spectral":
        # MSE + radially-averaged power spectrum matching. Directly targets
        # the "deterministic nets lose power at fine scales" problem.
        ws = kw.get("w_spec", 1.0)
        def f(p, t, m):
            base = _m((p - t) ** 2, m)
            pf = torch.fft.rfft2(p * m)
            tf = torch.fft.rfft2(t * m)
            pp = (pf.real ** 2 + pf.imag ** 2 + 1e-8).mean(dim=0)
            tp = (tf.real ** 2 + tf.imag ** 2 + 1e-8).mean(dim=0)
            return base + ws * (torch.log(pp) - torch.log(tp)).abs().mean()

    else:
        raise ValueError(f"unknown LOSS_NAME '{name}'")

    return f


# ======================================================================
# MODEL + TRAIN
# ======================================================================

def build_and_run(args, cache_path, out_maps, loss_name, loss_kw):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"    device: {dev} | loss: {loss_name} {loss_kw}")

    z = np.load(cache_path, allow_pickle=False)
    X, y, doy, wid = z["X"], z["y"], z["doy"], z["wid"]
    year, mask, is_test = z["year"], z["mask"], z["is_test"]
    clat, clon = z["clat"], z["clon"]
    flat_lat, flat_lon = z["flat_lat"], z["flat_lon"]
    n_win = len(z["window_names"]); H, W = len(flat_lat), len(flat_lon)
    n_var = X.shape[1]

    gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
    gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
    gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
    samp_np = np.stack([gxx, gyy], -1).astype(np.float32)[None]
    assert np.abs(samp_np).max() <= 1.0, "fine grid outside coarse box"

    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static_np = np.stack([mask.astype(DTYPE),
                          np.broadcast_to(lat2, (H, W)).astype(DTYPE),
                          np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(self, ci, co, drop=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                L.append(nn.Dropout2d(drop))
            self.f = nn.Sequential(*L)
        def forward(self, x):
            return self.f(x)

    class MWUNet(nn.Module):
        def __init__(self, n_var, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            self.emb = nn.Embedding(n_win, emb)
            self.enc_c1 = Block(n_var + emb, base * 2)
            self.enc_c2 = Block(base * 2, base * 2)
            self.inp = Block(base * 2 + 3, base)
            self.d1 = Block(base, base * 2, drop)
            self.d2 = Block(base * 2, base * 4, drop)
            self.bott = Block(base * 4, base * 4, drop)
            self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.du2 = Block(base * 4, base * 2, drop)
            self.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.du1 = Block(base * 2, base)
            self.head = nn.Conv2d(base, 1, 1)
            self.pool = nn.MaxPool2d(2)
        def forward(self, xc, wid, samp, static):
            b, _, h, w = xc.shape
            e = self.emb(wid)[:, :, None, None].expand(-1, -1, h, w)
            c = self.enc_c2(self.enc_c1(torch.cat([xc, e], 1)))
            f = F.grid_sample(c, samp.expand(b, -1, -1, -1),
                              mode="bilinear", align_corners=True)
            f = torch.cat([f, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = self.inp(f); e1 = self.d1(self.pool(e0)); e2 = self.d2(self.pool(e1))
            u = self.du2(torch.cat([self.u2(self.bott(e2)), e1], 1))
            u = self.du1(torch.cat([self.u1(u), e0], 1))
            return self.head(u)[:, :, :H0, :W0].squeeze(1)

    samp = torch.tensor(samp_np).to(dev)
    stat = torch.tensor(static_np).to(dev)

    def skill_acc(p, t, fin):
        se_m = np.where(fin, (t - p) ** 2, np.nan)
        se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
            rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rmse_c > 1e-6
            skill = np.where(ok, 1 - rmse_m / np.where(ok, rmse_c, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    def std_ratio(p, t, fin):
        """Diagnostic only -- reported, never optimised."""
        pv, tv = p[fin], t[fin]
        return float(pv.std() / tv.std()) if tv.std() > 0 else np.nan

    def train_one(tr_idx, va_idx, Xa, ya, max_epochs):
        Xt = torch.tensor(Xa)
        yt = torch.tensor(np.nan_to_num(ya))
        widt = torch.tensor(wid)
        fin = torch.tensor((np.isfinite(ya) & mask[None]).astype(DTYPE))

        tstd = float(np.nanstd(ya[tr_idx][np.isfinite(ya[tr_idx])]))
        crit = make_loss(loss_name, tstd, **loss_kw)

        def dl(idx, sh):
            return DataLoader(TensorDataset(Xt[idx], widt[idx], yt[idx], fin[idx]),
                              batch_size=args.batch, shuffle=sh)

        tr_dl, va_dl = dl(tr_idx, True), dl(va_idx, False)
        model = MWUNet(n_var, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xb, wb, yb, mb in tr_dl:
                xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                loss = crit(model(xb, wb, samp, stat), yb, mb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
            sched.step()

            model.eval()
            vl, n = 0.0, 0
            with torch.no_grad():
                for xb, wb, yb, mb in va_dl:
                    xb, wb, yb, mb = [t.to(dev) for t in (xb, wb, yb, mb)]
                    vl += float(crit(model(xb, wb, samp, stat), yb, mb)) * len(xb)
                    n += len(xb)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, Xa):
        Xt = torch.tensor(Xa); widt = torch.tensor(wid)
        out = []
        model.eval()
        with torch.no_grad():
            for a in range(0, len(idx), args.batch):
                j = idx[a:a + args.batch]
                out.append(model(Xt[j].to(dev), widt[j].to(dev), samp, stat).cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    blocks = np.array_split(nontest_years, args.folds)
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming: folds {sorted(done)} cached ({resume_path})")

    with stage(f"Rotating-year CV: {args.folds} folds over {len(nontest_years)} years"):
        for fi, val_years in enumerate(blocks):
            if fi in done:
                r = done[fi]
                print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | ACC {r['acc']:.3f}")
                continue
            val_years = set(val_years.tolist())
            va_i = np.isin(year, list(val_years)) & ~is_test
            tr_i = ~np.isin(year, list(val_years)) & ~is_test
            Xa, ya, _ = anomalise_fold(X, y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0],
                                          Xa, ya, args.epochs)
            p = predict(model, np.where(va_i)[0], Xa)
            t = ya[va_i]
            fin = np.isfinite(y[va_i]) & mask[None]
            sk, ac = skill_acc(p, t, fin)
            ms, ma = float(np.nanmean(sk[mask])), float(np.nanmean(ac[mask]))
            sr = std_ratio(p, t, fin)
            done[fi] = {"skill": ms, "acc": ma, "std_ratio": sr,
                        "val_years": sorted(val_years), "epochs": int(eps),
                        "vloss": float(vloss)}
            with open(resume_path, "w") as fh:
                json.dump({str(k): v for k, v in done.items()}, fh, indent=2)
            print(f"    fold {fi} val {sorted(val_years)}: skill {ms:+.3f} | "
                  f"ACC {ma:.3f} | std_ratio {sr:.3f} | {eps} ep   [saved]", flush=True)

        ks = [i for i in range(args.folds) if i in done]
        fs = [done[i]["skill"] for i in ks]; fa = [done[i]["acc"] for i in ks]
        fr = [done[i].get("std_ratio", np.nan) for i in ks]
        print(f"\n    [{loss_name}]  CV skill {np.mean(fs):+.4f} +/- {np.std(fs):.4f}"
              f" | CV ACC {np.mean(fa):.4f} +/- {np.std(fa):.4f}"
              f" | std_ratio {np.nanmean(fr):.3f}")

    # ---------- final model -> test ----------
    with stage("Final model on all non-test years -> test"):
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = (~is_test) & ~va_i
        Xa, ya, _ = anomalise_fold(X, y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0], Xa, ya, args.epochs)

        te_i = np.where(is_test)[0]
        p = predict(model, te_i, Xa)
        t = ya[is_test]
        fin = np.isfinite(y[is_test]) & mask[None]

        import xarray as xr
        wid_te = wid[is_test]; data_vars = {}
        print(f"    trained {eps} ep")
        summary = {}
        for w in range(n_win):
            sm = wid_te == w
            if sm.sum() == 0:
                continue
            sk, ac = skill_acc(p[sm], t[sm], fin[sm])
            sr = std_ratio(p[sm], t[sm], fin[sm])
            wn = str(z["window_names"][w])
            data_vars[f"{wn}_skill"] = (("lat", "lon"), sk)
            data_vars[f"{wn}_acc"] = (("lat", "lon"), ac)
            summary[wn] = {"skill": float(np.nanmean(sk[mask])),
                           "acc": float(np.nanmean(ac[mask])),
                           "std_ratio": sr,
                           "pct_pos": float(100 * np.nanmean(sk[mask] > 0))}
            print(f"    test {wn:>8}: skill {summary[wn]['skill']:+.4f} | "
                  f"ACC {summary[wn]['acc']:.4f} | std_ratio {sr:.3f} | "
                  f"{summary[wn]['pct_pos']:.0f}% cells+")

        out = xr.Dataset(data_vars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["loss"] = f"{loss_name} {loss_kw}"
        out.attrs["windows"] = ", ".join(f"{n}:{lo}-{hi}" for n, lo, hi in WINDOWS)
        out.to_netcdf(out_maps)
        torch.save({"state": model.state_dict(), "args": ARG_DICT}, out_maps.replace(".nc", ".pt"))

        # append to a cross-loss comparison table
        tbl = "loss_comparison.json"
        allr = json.load(open(tbl)) if os.path.exists(tbl) else {}
        allr[tag] = {"cv_skill": float(np.mean(fs)), "cv_acc": float(np.mean(fa)),
                     "cv_std_ratio": float(np.nanmean(fr)), "test": summary}
        json.dump(allr, open(tbl, "w"), indent=2)
        print(f"    -> {out_maps} (+ .pt); comparison appended to {tbl}")


# ======================================================================
if args.cmd == "prepare":
    prepare(ds_ecmv, ds_imd, CACHE)          # noqa: F821
else:
    if not os.path.exists(CACHE):
        raise SystemExit(f"run prepare first ({CACHE} missing)")
    build_and_run(args, CACHE, OUT_MAPS, LOSS_NAME, LOSS_KW)

    # print the running cross-loss comparison
    tbl = "loss_comparison.json"
    if os.path.exists(tbl):
        allr = json.load(open(tbl))
        print(f"\n{'loss':>22} {'CV skill':>10} {'CV ACC':>8} {'std':>6} "
              f"{'w3_4 skill':>11} {'w3_4 ACC':>9}")
        for k, v in sorted(allr.items()):
            w = v["test"].get("week3_4", {})
            print(f"{k:>22} {v['cv_skill']:>+10.4f} {v['cv_acc']:>8.4f} "
                  f"{v['cv_std_ratio']:>6.3f} {w.get('skill', float('nan')):>+11.4f} "
                  f"{w.get('acc', float('nan')):>9.4f}")

    device: mps | loss: mae {}
[ ] Rotating-year CV: 5 folds over 17 years ...


/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/2039584478.py:384: RuntimeWarning: Mean of empty slice
  rmse_m = np.sqrt(np.nanmean(se_m, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/2039584478.py:385: RuntimeWarning: Mean of empty slice
  rmse_c = np.sqrt(np.nanmean(se_c, axis=0))
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/2039584478.py:388: RuntimeWarning: Mean of empty slice
  tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
/var/folders/h4/hvzvbp993dx41cy379v1lblm0000gn/T/ipykernel_22186/2039584478.py:389: RuntimeWarning: Mean of empty slice
  pm = np.nanmean(np.where(fin, p, np.nan), axis=0)


    fold 0 val [2005, 2006, 2007, 2008]: skill +0.015 | ACC 0.226 | std_ratio 0.292 | 29 ep   [saved]


KeyboardInterrupt: 